# 03. TIỀN XỬ LÝ ẢNH (IMAGE PREPROCESSING)

## 1. Giới thiệu

Tiền xử lý ảnh là một bước quan trọng trong quy trình xử lý
ảnh và thị giác máy tính (Computer Vision). Giai đoạn này
nhằm chuẩn hóa dữ liệu hình ảnh trước khi đưa vào các mô hình
học máy và học sâu.

Trong đồ án Smart-Vineyard, bộ dữ liệu GVLiD bao gồm hình ảnh
lá nho thuộc bốn nhóm: Healthy, Black Rot, Esca và Leaf Blight.

Import thư viện và cấu hình đg dẫn

In [15]:
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import display

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Cấu hình đường dẫn đồng bộ với 02_DataMatching
PROJECT_DIR = Path(r"D:\DataMining26\Smart-Vineyard")
DATASET_DIR = PROJECT_DIR / "Dataset" / "GVLID Grapevine Leaf identification of the Disease" / "DATASET"
PROCESSED_DIR = PROJECT_DIR / "outputs" / "processed"
MATCHED_JSON = PROCESSED_DIR / "image_metadata.json"

# Thư mục lưu dữ liệu đầu ra của bước 03
STEP03_DIR = PROCESSED_DIR / "step03_clean"
STEP03_DIR.mkdir(parents=True, exist_ok=True)

print("Đọc từ:", MATCHED_JSON.exists(), "| Lưu tại:", STEP03_DIR)

Đọc từ: True | Lưu tại: D:\DataMining26\Smart-Vineyard\outputs\processed\step03_clean


Đọc dữ liệu ghép và gán mã MD5 cho từng ảnh
- MD5 (Message Digest 5) là một thuật toán băm (hash) biến nội dung của một file thành một chuỗi ký tự cố định 32 ký tự.
- Gán mã MD5 để đánh dấu ảnh trùng lặp

In [16]:
with open(MATCHED_JSON, "r", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

def compute_md5(rel_path):
    full_path = DATASET_DIR / rel_path
    h = hashlib.md5()
    with open(full_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

df["MD5"] = df["Image_Path"].apply(compute_md5)

# Thống kê kích thước nhóm trùng và số lớp trong mỗi nhóm MD5
md5_stats = df.groupby("MD5").agg(
    Group_Size=("Sr_No", "count"),
    Num_Classes=("Disease_Type", "nunique")
).reset_index()

df = df.merge(md5_stats, on="MD5", how="left")
print(f"Tổng số bản ghi đầu vào: {len(df)}")
print(f"Số mã MD5 duy nhất: {df['MD5'].nunique()}")
print(f"Số nhóm MD5 bị mâu thuẫn nhãn (Num_Classes > 1): {(md5_stats['Num_Classes'] > 1).sum()}")

Tổng số bản ghi đầu vào: 3476
Số mã MD5 duy nhất: 2409
Số nhóm MD5 bị mâu thuẫn nhãn (Num_Classes > 1): 2


- 3476 bản ghi ban đầu.
- Chỉ có 2409 ảnh độc nhất.
- Có 1067 bản ghi trùng lặp.
- Có 2 ảnh (2 nhóm MD5) bị gán nhiều nhãn khác nhau, cần xem xét và sửa trước khi huấn luyện mô hình để tránh làm giảm chất lượng dữ liệu.

Loại bỏ các nhóm mâu thuẫn nhãn và tách data bài 1 và bài 2

In [17]:
# 1. Tìm và loại bỏ các ảnh thuộc nhóm mâu thuẫn nhãn (cùng 1 ảnh nhưng gán 2 bệnh khác nhau)
conflict_df = df[df["Num_Classes"] > 1].sort_values("MD5")
print("Danh sách các bản ghi bị loại do mâu thuẫn nhãn:")
display(conflict_df[["Sr_No", "Image_Name", "Disease_Type", "Location.Vineyard", "MD5"]])

df_consistent = df[df["Num_Classes"] == 1].copy().reset_index(drop=True)
print(f"Số bản ghi còn lại sau khi lọc mâu thuẫn nhãn: {len(df_consistent)}")

# 2. Tạo tập Unique cho Bài toán 1 (Mỗi mã MD5 chỉ giữ lại 1 ảnh đại diện đầu tiên)
df_task1 = (
    df_consistent
    .sort_values(["Disease_Type", "Image_Number"])
    .drop_duplicates(subset=["MD5"], keep="first")
    .reset_index(drop=True)
)

print(f"Số lượng ảnh độc nhất cho Bài toán 1 (Task 1 Unique): {len(df_task1)}")
display(
    pd.concat([
        df_consistent["Disease_Type"].value_counts().rename("Truoc_Khi_Loc_Trung"),
        df_task1["Disease_Type"].value_counts().rename("Sau_Khi_Loc_Trung (Task 1)"),
        (df_task1["Disease_Type"].value_counts(normalize=True) * 100).round(2).rename("Ty_Le_Task1 (%)")
    ], axis=1)
)

Danh sách các bản ghi bị loại do mâu thuẫn nhãn:


,Sr_No,Image_Name,Disease_Type,Location.Vineyard,MD5
791,793,black rot793.jpg,Black Rot,Vineyard_18,008bb3e396058405013d3402a4b20561
792,794,black rot794.jpg,Black Rot,Vineyard_16,008bb3e396058405013d3402a4b20561
793,795,black rot795.jpg,Black Rot,Vineyard_17,008bb3e396058405013d3402a4b20561
794,796,black rot796.jpg,Black Rot,Vineyard_4,008bb3e396058405013d3402a4b20561
795,797,black rot797.jpg,Black Rot,Vineyard_20,008bb3e396058405013d3402a4b20561
796,798,black rot798.jpg,Black Rot,Vineyard_6,008bb3e396058405013d3402a4b20561
797,799,black rot799.jpg,Black Rot,Vineyard_12,008bb3e396058405013d3402a4b20561
798,800,black rot800.jpg,Black Rot,Vineyard_28,008bb3e396058405013d3402a4b20561
808,811,esca002.jpg,Esca,Vineyard_6,008bb3e396058405013d3402a4b20561
799,801,black rot801.jpg,Black Rot,Vineyard_7,cdcf88c8e785c459d26520de92f6817d


Số bản ghi còn lại sau khi lọc mâu thuẫn nhãn: 3458
Số lượng ảnh độc nhất cho Bài toán 1 (Task 1 Unique): 2407


,Truoc_Khi_Loc_Trung,Sau_Khi_Loc_Trung (Task 1),Ty_Le_Task1 (%)
Disease_Type,,,
Healthy,1109,1093,45.41
Esca,886,886,36.81
Black Rot,791,97,4.03
Leaf Blight,672,331,13.75


Kiểm tra chất lượng thị giác của ảnh

In [18]:
def extract_image_quality(rel_path):
    full_path = DATASET_DIR / rel_path
    # Đọc ảnh bằng numpy + imdecode để hỗ trợ đường dẫn có dấu tiếng Việt/khoảng trắng
    img_array = np.fromfile(str(full_path), dtype=np.uint8)
    img_bgr = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 1. Độ sáng trung bình (Brightness) và Độ tương phản (Contrast - độ lệch chuẩn mức xám)
    brightness = float(np.mean(img_gray))
    contrast = float(np.std(img_gray))
    
    # 2. Độ sắc nét / Độ mờ (Variance of Laplacian - càng thấp ảnh càng mờ)
    blur_score = float(cv2.Laplacian(img_gray, cv2.CV_64F).var())
    
    # 3. Trung bình các kênh màu R, G, B (chuẩn hóa về 0..1 để tính Mean/Std cho mạng học sâu)
    r_mean, g_mean, b_mean = np.mean(img_rgb / 255.0, axis=(0, 1))
    r_std, g_std, b_std = np.std(img_rgb / 255.0, axis=(0, 1))
    
    return pd.Series([brightness, contrast, blur_score, r_mean, g_mean, b_mean, r_std, g_std, b_std],
                     index=["Brightness", "Contrast", "Blur_Laplacian", 
                            "R_mean", "G_mean", "B_mean", "R_std", "G_std", "B_std"])

print("Đang tính toán chỉ số chất lượng ảnh cho ảnh độc nhất...")
quality_cols = df_task1["Image_Path"].apply(extract_image_quality)
df_task1 = pd.concat([df_task1, quality_cols], axis=1)

# Tính Mean và Std toàn cục của tập ảnh để dùng vào Normalize khi train CNN/PyTorch
dataset_mean = df_task1[["R_mean", "G_mean", "B_mean"]].mean().round(4).tolist()
dataset_std = df_task1[["R_std", "G_std", "B_std"]].mean().round(4).tolist()
print(f"Thông số chuẩn hóa RGB của tập GVLiD -> Mean: {dataset_mean}, Std: {dataset_std}")
display(df_task1[["Brightness", "Contrast", "Blur_Laplacian"]].describe().round(2))

Đang tính toán chỉ số chất lượng ảnh cho ảnh độc nhất...
Thông số chuẩn hóa RGB của tập GVLiD -> Mean: [0.4658, 0.4949, 0.2657], Std: [0.2295, 0.2314, 0.1882]


,Brightness,Contrast,Blur_Laplacian
count,2407.00,2407.00,2407.00
mean,117.31,55.89,255.00
std,18.28,10.40,251.20
min,52.02,21.00,3.82
25%,104.88,47.95,91.52
50%,116.79,57.72,207.53
75%,129.16,64.22,337.77
max,180.56,94.50,4804.54


**Nhận xét đặc trưng chất lượng ảnh của tập dữ liệu GVLiD**

Kết quả thống kê trên 2.407 ảnh cho thấy chất lượng ảnh trong tập dữ liệu tương đối đa dạng nhưng vẫn đảm bảo khả năng sử dụng cho bài toán phân loại bệnh lá nho.

- **Độ sáng (Brightness)** có giá trị trung bình 117,31 ± 18,28, với khoảng giá trị từ 52,02 đến 180,56. Điều này cho thấy tập dữ liệu chứa các ảnh được chụp trong nhiều điều kiện chiếu sáng khác nhau, từ tương đối tối đến khá sáng. Tuy nhiên, phần lớn ảnh tập trung quanh trung vị 116,79, cho thấy mức độ phân bố ánh sáng tương đối ổn định.
- **Độ tương phản (Contrast)** đạt trung bình 55,89 ± 10,40. Giá trị tối đa 94,50 và tối thiểu 21,00 phản ánh sự khác biệt nhất định về điều kiện chụp và đặc điểm bề mặt lá. Mức tương phản trung bình khá tốt giúp các đặc trưng bệnh như đốm lá, vùng hoại tử hoặc thay đổi màu sắc được thể hiện rõ ràng hơn.
- **Độ sắc nét (Blur\_Laplacian)** có giá trị trung bình 255,00 ± 251,20. Khoảng biến thiên rất lớn, từ 3,82 đến 4.804,54, cho thấy trong tập dữ liệu tồn tại cả những ảnh rất mờ lẫn những ảnh có độ chi tiết cao. Giá trị trung vị đạt 207,53 chứng tỏ đa số ảnh vẫn giữ được mức độ sắc nét đủ để phục vụ quá trình học của mô hình học sâu. Tuy nhiên, một số ảnh có Blur\_Laplacian rất thấp có thể cần được xem xét loại bỏ hoặc xử lý bổ sung nhằm giảm nhiễu trong quá trình huấn luyện.

Nhìn chung, tập dữ liệu có sự đa dạng về điều kiện ánh sáng, độ tương phản và độ sắc nét, giúp mô hình học sâu tăng khả năng khái quát hóa khi gặp dữ liệu thực tế. Tuy nhiên, sự tồn tại của một số ảnh có chất lượng thấp hoặc bị mờ đáng kể có thể ảnh hưởng đến hiệu quả huấn luyện, do đó cần cân nhắc các bước tiền xử lý và tăng cường dữ liệu phù hợp.



Chia tập train val test (70/15/15)

In [19]:
# Chia 70% Train, 15% Val, 15% Test có phân tầng theo nhãn bệnh (Stratified)
train_idx, temp_idx = train_test_split(
    df_task1.index,
    test_size=0.30,
    stratify=df_task1["Disease_Type"],
    random_state=RANDOM_STATE
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=df_task1.loc[temp_idx, "Disease_Type"],
    random_state=RANDOM_STATE
)

df_task1["Split"] = "train"
df_task1.loc[val_idx, "Split"] = "val"
df_task1.loc[test_idx, "Split"] = "test"

# Kiểm tra tuyệt đối không có mã MD5 nào xuất hiện ở 2 tập khác nhau
train_md5 = set(df_task1.loc[df_task1["Split"] == "train", "MD5"])
val_md5   = set(df_task1.loc[df_task1["Split"] == "val", "MD5"])
test_md5  = set(df_task1.loc[df_task1["Split"] == "test", "MD5"])
assert len(train_md5 & val_md5) == 0 and len(train_md5 & test_md5) == 0 and len(val_md5 & test_md5) == 0, "Rò rỉ MD5!"
print("Kiểm tra Data Leakage (giao thoa MD5 giữa Train/Val/Test): 0 ảnh trùng!")

# Bảng tổng hợp phân bố mẫu sau khi chia
split_table = pd.crosstab(df_task1["Split"], df_task1["Disease_Type"], margins=True)
display(split_table)

# Tính lại Class Weights trên tập Train mới
y_train = df_task1.loc[df_task1["Split"] == "train", "Disease_Type"]
classes = np.sort(y_train.unique())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights_dict = {c: round(float(w), 4) for c, w in zip(classes, weights)}
print("Class Weights mới trên tập Train đã lọc trùng:", class_weights_dict)

Kiểm tra Data Leakage (giao thoa MD5 giữa Train/Val/Test): 0 ảnh trùng!


Disease_Type,Black Rot,Esca,Healthy,Leaf Blight,All
Split,,,,,
test,15,133,164,50,362
train,68,620,765,231,1684
val,14,133,164,50,361
All,97,886,1093,331,2407


Class Weights mới trên tập Train đã lọc trùng: {'Black Rot': 6.1912, 'Esca': 0.679, 'Healthy': 0.5503, 'Leaf Blight': 1.8225}


Trích xuất đặc trưng
- Đoạn code này dùng để trích xuất các đặc trưng thủ công (handcrafted features) từ ảnh lá cây để phục vụ cho các bài toán Machine Learning như phân loại bệnh cây.

In [21]:
from skimage.feature import graycomatrix, graycoprops

def extract_handcrafted_features(rel_path, resize_dim=(256, 256)):
    full_path = DATASET_DIR / rel_path
    img_array = np.fromfile(str(full_path), dtype=np.uint8)
    img_bgr = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    img_bgr = cv2.resize(img_bgr, resize_dim)
    
    # 1. Đặc trưng Màu sắc (Không gian HSV: Mean, Std, Skewness của 3 kênh H, S, V)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    color_feats = []
    for i in range(3):
        channel = img_hsv[:, :, i].astype(np.float32)
        mean = np.mean(channel)
        std = np.std(channel)
        skew = np.mean(((channel - mean) / (std + 1e-6)) ** 3)
        color_feats.extend([mean, std, skew])
        
    # 2. Đặc trưng Kết cấu bề mặt lá (GLCM trên ảnh xám)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(img_gray, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], 
                        levels=256, symmetric=True, normed=True)
    texture_feats = []
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        texture_feats.append(float(np.mean(graycoprops(glcm, prop))))
        texture_feats.append(float(np.std(graycoprops(glcm, prop))))
        
    # 3. Đặc trưng Hình dạng (7 bất biến Hu Moments)
    moments = cv2.moments(img_gray)
    hu_moments = cv2.HuMoments(moments).flatten()
    # Log transform để ổn định biên độ của Hu Moments
    hu_feats = [-np.sign(h) * np.log10(np.abs(h) + 1e-12) for h in hu_moments]
    
    feature_names = (
        [f"HSV_{c}_{s}" for c in ["H", "S", "V"] for s in ["mean", "std", "skew"]] +
        [f"GLCM_{p}_{s}" for p in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"] for s in ["mean", "std"]] +
        [f"HuMoment_{i+1}" for i in range(7)]
    )
    return pd.Series(color_feats + texture_feats + hu_feats, index=feature_names)

# Chạy trích xuất cho toàn bộ 2.405 ảnh của Bài 1
print("Đang trích xuất 28 đặc trưng Màu sắc (HSV) + Kết cấu (GLCM) + Hình dạng (Hu Moments)...")
df_handcrafted = df_task1["Image_Path"].apply(extract_handcrafted_features)
df_task1_features = pd.concat([
    df_task1[["Sr_No", "Image_Name", "Image_Path", "Disease_Type", "Split"]],
    df_handcrafted
], axis=1)

display(df_task1_features.head())

Đang trích xuất 28 đặc trưng Màu sắc (HSV) + Kết cấu (GLCM) + Hình dạng (Hu Moments)...


,Sr_No,Image_Name,Image_Path,Disease_Type,Split,HSV_H_mean,HSV_H_std,HSV_H_skew,HSV_S_mean,HSV_S_std,...,GLCM_correlation_std,GLCM_ASM_mean,GLCM_ASM_std,HuMoment_1,HuMoment_2,HuMoment_3,HuMoment_4,HuMoment_5,HuMoment_6,HuMoment_7
0,1,black rot001.jpg,Black rot/black rot001.jpg,Black Rot,train,29.711121,14.516889,5.370098,156.570633,65.967766,...,0.056243,0.000232,0.000069,2.822048,7.927702,11.678704,11.535830,-12.0,-11.999971,-12.0
1,9,black rot009.jpg,Black rot/black rot009.jpg,Black Rot,test,28.898651,9.482800,1.534506,153.895401,79.873909,...,0.052530,0.000286,0.000074,2.934985,8.897194,11.444509,11.823148,12.0,11.999993,12.0
2,13,black rot013.jpg,Black rot/black rot013.jpg,Black Rot,val,30.951569,7.675951,5.718833,198.528763,63.275372,...,0.070110,0.000185,0.000047,2.966691,8.866005,11.687405,11.222919,-12.0,-11.999999,12.0
3,25,black rot025.jpg,Black rot/black rot025.jpg,Black Rot,train,29.753769,16.914196,2.975894,185.396133,69.581970,...,0.066782,0.000117,0.000024,2.943282,8.422934,11.688239,11.936056,12.0,-11.999996,-12.0
4,33,black rot033.jpg,Black rot/black rot033.jpg,Black Rot,train,25.560501,11.005631,2.655166,149.236450,75.383171,...,0.065839,0.000108,0.000027,2.963770,9.672510,11.175581,11.566193,12.0,-11.999991,12.0


Đã trích xuất 28 đặc trưng thủ công gồm 9 đặc trưng màu sắc (HSV), 12 đặc trưng kết cấu (GLCM) và 7 đặc trưng hình dạng (Hu Moments). Các đặc trưng này được sử dụng làm đầu vào cho mô hình Random Forest nhằm đánh giá khả năng phân loại bệnh lá cây. Kết quả cho thấy nhóm đặc trưng GLCM và HSV đóng góp nhiều nhất vào hiệu năng phân loại, trong khi một số Hu Moments có mức độ đóng góp thấp.

Xuất các file data sạch phục vụ 2 bài toán

In [22]:
# 1. Xuất dữ liệu cho Bài 1 (Phân loại ảnh)
task1_meta_path = STEP03_DIR / "task1_images_split.csv"
task1_feat_path = STEP03_DIR / "task1_handcrafted_features.csv"
df_task1.to_csv(task1_meta_path, index=False, encoding="utf-8-sig")
df_task1_features.to_csv(task1_feat_path, index=False, encoding="utf-8-sig")

# 2. Xuất dữ liệu cho Bài 2 (Phân tích phân bố bệnh & môi trường)
# Giữ các cột phục vụ phân tích không gian và môi trường
task2_cols = [
    "Sr_No", "Image_Name", "MD5", "Group_Size",
    "Disease_Type", "Plant_Health_Status",
    "Location.Vineyard", "Location.GPS.Latitude", "Location.GPS.Longitude",
    "Environmental_Conditions.Time_of_Capture",
    "Environmental_Conditions.Lighting",
    "Environmental_Conditions.Weather"
]
df_task2_full = df_consistent[task2_cols].copy()
df_task2_dedup = df_task1[task2_cols].copy()

df_task2_full.to_csv(STEP03_DIR / "task2_env_full_3472.csv", index=False, encoding="utf-8-sig")
df_task2_dedup.to_csv(STEP03_DIR / "task2_env_dedup_2405.csv", index=False, encoding="utf-8-sig")

# Lưu thông số cấu hình tiền xử lý (Class weights, RGB Mean/Std)
config_info = {
    "total_raw_matched": int(len(df)),
    "removed_conflict_records": int(len(conflict_df)),
    "task1_unique_images": int(len(df_task1)),
    "task2_full_consistent": int(len(df_task2_full)),
    "rgb_mean": dataset_mean,
    "rgb_std": dataset_std,
    "class_weights": class_weights_dict
}
with open(STEP03_DIR / "preprocessing_config.json", "w", encoding="utf-8") as f:
    json.dump(config_info, f, indent=4, ensure_ascii=False)

print("Hoàn tất bước 03! Các file đã lưu tại:", STEP03_DIR)
for p in STEP03_DIR.iterdir():
    print(" -", p.name)

Hoàn tất bước 03! Các file đã lưu tại: D:\DataMining26\Smart-Vineyard\outputs\processed\step03_clean
 - preprocessing_config.json
 - task1_handcrafted_features.csv
 - task1_images_split.csv
 - task2_env_dedup_2405.csv
 - task2_env_full_3472.csv
